# Demand Forecasting - Data Preprocessing (Daily Grain)

This notebook turns the raw warehouse pick-transaction exports into a clean,
**daily** SKU-level table ready for model training.

**Changed from weekly to daily (2026-10-01, per team decision):** the team
wants to capture day-of-week effects (weekends being slower than weekdays),
which a weekly total completely averages away. This version has one row per
(SKU, calendar day) instead of one row per (SKU, week), and adds explicit
day-of-week / weekend features.

**Before running:** put your raw CSV files (the main export and the archive
export) into a folder called `raw_data/` in the same location as this
notebook. To add new data later, just drop the new file into `raw_data/` and
rerun every cell - nothing else needs to change.

Run the cells top to bottom with **Shift+Enter**.

In [1]:
import glob
import pandas as pd
import numpy as np

## Config

These are the settings you might actually want to change. Everything below
this cell is the pipeline logic - you shouldn't need to touch it.

In [2]:
RAW_DATA_DIR = "raw_data"
RAW_DATA_GLOB = f"{RAW_DATA_DIR}/*.csv"   # picks up every CSV dropped in this folder
OUTPUT_FILE = "sku_day_panel.csv"

KEEP_COLS = ["NO", "CREATE_DT", "FROM_ITEM", "FROM_DESCR", "CASES_PER_PALLET",
             "CUST_NAME", "FROM_ZONE", "FROM_UOM"]

# Which warehouse zone a pick happened in tells you which of two different
# physical operations it was. WZPCK is the case-pick face (mostly single-case
# picks). The WZGNx / WZPKG zones are reserve storage, where whole pallets get
# pulled instead.
CASE_PICK_ZONE = "WZPCK"
PALLET_PULL_ZONES = {"WZGNA", "WZGNB", "WZGNC", "WZGND", "WZGNE", "WZGNF",
                      "WZGNG", "WZGNH", "WZGNJ", "WZPKG"}

# Tag every output row with where this number came from and what it isn't.
SOURCE_LABEL = "pick_proxy_filtered"
CAVEAT = (
    "Every count in this file comes from WAREHOUSE PICK TRANSACTIONS, "
    "filtered to rows with a non-blank CUST_NAME (customer-order picks). "
    "This is a PROXY for real demand, not a direct order-line extract: the "
    "pick timestamp (CREATE_DT) marks when the warehouse released the work, "
    "not when the customer actually placed the order. Replace this source "
    "with a true order-line extract (an order-received timestamp) once one "
    "exists."
)

# Daily lags: yesterday, the last few days, same day last 1-4 weeks, and
# same day last year. lag_7 (this day last week) is especially useful at
# daily grain because of day-of-week seasonality.
LAG_DAYS = [1, 2, 3, 7, 14, 21, 28, 364]

# 7/28/56 days = roughly 1/4/8 weeks - mirrors the 4- and 8-week windows
# from the old weekly version, plus a tight 7-day window.
ROLLING_WINDOWS = [7, 28, 56]

# How many of the most recent complete days to hold out as the final test
# set - one fiscal quarter (13 weeks * 7 days = 91 days).
TEST_DAYS = 91

## Step 1: Load one raw file and filter to real customer demand

A row only counts as real demand if it has a non-blank `CUST_NAME` - this is
what separates actual customer pick demand from internal warehouse moves
(raw-material replenishment, internal transfers, etc). See the project doc
`data-finding-real-demand-window.md` for how this was discovered.

**Note:** this parses `CREATE_DT` two ways - a fast pass for the expected
format, then a slower, more flexible fallback only for any rows that don't
match it (handles the case where Excel silently reformats dates if a file
is ever opened and re-saved there).

In [3]:
def parse_create_dt(raw_dates: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(raw_dates, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    failed = parsed.isna() & raw_dates.notna()
    if failed.any():
        print(f"  {failed.sum():,} of {len(raw_dates):,} dates didn't match the expected "
              f"format - re-parsing those with a slower, more flexible parser "
              f"(likely an Excel re-save changed their format).")
        parsed.loc[failed] = pd.to_datetime(
            raw_dates.loc[failed], format="mixed", dayfirst=False
        )
    return parsed


def load_and_filter(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=KEEP_COLS, dtype={"FROM_ITEM": str, "NO": str})
    df["CREATE_DT"] = parse_create_dt(df["CREATE_DT"])

    before = len(df)
    has_customer = df["CUST_NAME"].notna() & (df["CUST_NAME"].str.strip() != "")
    df = df[has_customer].drop(columns=["CUST_NAME"])
    dropped = before - len(df)
    print(f"{path}: dropped {dropped:,} non-customer rows ({dropped/before:.1%} of {before:,}), "
          f"kept {len(df):,}")
    return df

## Step 2: Load and combine every file in `raw_data/`

Automatically finds every CSV in the folder, filters each one, and checks
that no transaction ID (`NO`) appears in more than one file before combining
them.

In [4]:
def load_all_raw_files() -> pd.DataFrame:
    paths = sorted(glob.glob(RAW_DATA_GLOB))
    if not paths:
        raise FileNotFoundError(
            f"No CSV files found matching {RAW_DATA_GLOB} - "
            f"put the raw exports in {RAW_DATA_DIR}/ before running this notebook."
        )
    print(f"Found {len(paths)} raw file(s): {paths}")

    frames = [load_and_filter(p) for p in paths]
    combined = pd.concat(frames, ignore_index=True)

    before = len(combined)
    combined = combined.drop_duplicates(subset=["NO"], keep="first")
    dupes = before - len(combined)
    if dupes:
        print(f"WARNING: dropped {dupes:,} rows with a transaction ID (NO) that appeared "
              f"in more than one file - investigate before trusting this run's numbers.")
    else:
        print("No duplicate transaction IDs across files - confirmed safe to combine as-is.")

    return combined.drop(columns=["NO"])

combined = load_all_raw_files()
combined.head()

Found 2 raw file(s): ['raw_data/yowmspd2__picks_past_365days_ranon9_21_26.csv', 'raw_data/yowmspd2_archive_picks_past_365days_ranon9_21_26.csv']


raw_data/yowmspd2__picks_past_365days_ranon9_21_26.csv: dropped 21,791 non-customer rows (3.0% of 722,044), kept 700,253


raw_data/yowmspd2_archive_picks_past_365days_ranon9_21_26.csv: dropped 0 non-customer rows (0.0% of 1,434,288), kept 1,434,288


No duplicate transaction IDs across files - confirmed safe to combine as-is.


,CREATE_DT,FROM_ITEM,FROM_DESCR,FROM_UOM,CASES_PER_PALLET,FROM_ZONE
0,2025-10-22 11:18:15,20510118,VITFUS VIT D 4/3PKX150CT,CS,126,WZPCK
1,2025-10-22 11:20:05,20510118,VITFUS VIT D 4/3PKX150CT,CS,126,WZPCK
2,2025-10-22 11:20:30,20510118,VITFUS VIT D 4/3PKX150CT,CS,126,WZPCK
3,2025-10-28 03:42:51,20508856,AHLLD CLN BURST 8/33OZ,PL,54,WZGNA
4,2025-10-28 04:18:49,20508982,AHDC COMPLETE CARE TTHPASTE 6/6.0OZ 4PK,PL,120,WZNDK


## Step 3: Basic cleanup

Drop rows missing a date or SKU ID (unusable), and fill in placeholders for
anything else missing rather than throwing away a real transaction.

In [5]:
def clean_raw(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.dropna(subset=["CREATE_DT", "FROM_ITEM"])
    df["FROM_DESCR"] = df["FROM_DESCR"].fillna("UNKNOWN")
    df["CASES_PER_PALLET"] = df["CASES_PER_PALLET"].fillna(-1)
    df["FROM_ZONE"] = df["FROM_ZONE"].fillna("UNKNOWN")
    dropped = before - len(df)
    if dropped:
        print(f"Dropped {dropped} rows with missing date or item ID ({dropped/before:.2%})")
    return df

combined = clean_raw(combined)
combined.shape

(2134541, 6)

## Step 4: Build the daily SKU panel, split by warehouse process

Turns individual pick transactions into "how many times was SKU X picked on
day Y", filling in explicit zeros for days a SKU had no activity. Also
drops the first and last calendar day in the data, since each only has a
few hours of coverage (the raw export doesn't start/end exactly at
midnight) - same boundary-trimming idea as the old weekly version, just
applied per day.

Also classifies each pick by which physical operation it came from -
`case_pick` (the pick face, zone WZPCK) vs. `pallet_pull` (reserve storage,
zones WZGNA-WZGNJ/WZPKG) vs. `other` - and keeps those counts as separate
columns alongside the combined total.

In [6]:
def classify_process(zone: str) -> str:
    if zone == CASE_PICK_ZONE:
        return "case_pick"
    if zone in PALLET_PULL_ZONES:
        return "pallet_pull"
    return "other"


def build_sku_day_panel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["day"] = df["CREATE_DT"].dt.normalize()
    df["process"] = df["FROM_ZONE"].map(classify_process)

    data_min, data_max = df["CREATE_DT"].min(), df["CREATE_DT"].max()
    first_day, last_day = data_min.normalize(), data_max.normalize()
    full_day_lo = first_day if data_min == first_day else first_day + pd.Timedelta(days=1)
    full_day_hi = last_day if data_max == last_day else last_day - pd.Timedelta(days=1)

    panel = (
        df.groupby(["day", "FROM_ITEM"])
        .size()
        .rename("pick_lines")
        .reset_index()
    )

    breakdown = (
        df.groupby(["day", "FROM_ITEM", "process"])
        .size()
        .unstack("process", fill_value=0)
        .reindex(columns=["case_pick", "pallet_pull", "other"], fill_value=0)
    )
    breakdown.columns = [f"{c}_lines" for c in breakdown.columns]
    breakdown = breakdown.reset_index()

    panel = panel.merge(breakdown, on=["day", "FROM_ITEM"], how="left")

    all_days_seen = panel["day"].unique()
    full_days = [d for d in all_days_seen if full_day_lo <= d <= full_day_hi]
    dropped_days = sorted(set(all_days_seen) - set(full_days))
    if dropped_days:
        print(f"Dropped {len(dropped_days)} partial boundary day(s) from the panel: "
              f"{[str(d.date()) for d in dropped_days]}")

    all_skus = panel["FROM_ITEM"].unique()
    full_grid = pd.date_range(min(full_days), max(full_days), freq="D")
    full_grid = pd.MultiIndex.from_product(
        [full_grid, all_skus], names=["day", "FROM_ITEM"]
    ).to_frame(index=False)

    panel = full_grid.merge(panel, on=["day", "FROM_ITEM"], how="left")
    count_cols = ["pick_lines", "case_pick_lines", "pallet_pull_lines", "other_lines"]
    panel[count_cols] = panel[count_cols].fillna(0)
    return panel.sort_values(["FROM_ITEM", "day"]).reset_index(drop=True)

panel = build_sku_day_panel(combined)
panel.head()

Dropped 2 partial boundary day(s) from the panel: ['2025-09-21', '2026-09-21']


,day,FROM_ITEM,pick_lines,case_pick_lines,pallet_pull_lines,other_lines
0,2025-09-22,20000035,0.0,0.0,0.0,0.0
1,2025-09-23,20000035,0.0,0.0,0.0,0.0
2,2025-09-24,20000035,0.0,0.0,0.0,0.0
3,2025-09-25,20000035,0.0,0.0,0.0,0.0
4,2025-09-26,20000035,0.0,0.0,0.0,0.0


## Step 5: Build SKU-level attributes

One row per SKU: description, pack size, brand family, and a velocity tier
(A/B/C) based on how much of total pick volume that SKU accounts for
(standard 80%/95% ABC/Pareto cutoffs).

In [7]:
def build_sku_attributes(df: pd.DataFrame) -> pd.DataFrame:
    attrs = (
        df.groupby("FROM_ITEM")
        .agg(
            FROM_DESCR=("FROM_DESCR", lambda x: x.mode().iloc[0]),
            CASES_PER_PALLET=("CASES_PER_PALLET", lambda x: x.mode().iloc[0]),
        )
        .reset_index()
    )
    attrs["product_family"] = attrs["FROM_DESCR"].str.split().str[0]

    total_by_sku = df.groupby("FROM_ITEM").size().sort_values(ascending=False)
    cum_share = total_by_sku.cumsum() / total_by_sku.sum()
    tier = pd.cut(cum_share, bins=[-0.01, 0.80, 0.95, 1.0], labels=["A", "B", "C"])
    attrs = attrs.merge(
        tier.rename("velocity_tier").reset_index().rename(columns={"index": "FROM_ITEM"}),
        on="FROM_ITEM", how="left",
    )
    return attrs

attrs = build_sku_attributes(combined)
panel = panel.merge(attrs, on="FROM_ITEM", how="left")
panel.head()

,day,FROM_ITEM,pick_lines,case_pick_lines,pallet_pull_lines,other_lines,FROM_DESCR,CASES_PER_PALLET,product_family,velocity_tier
0,2025-09-22,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C
1,2025-09-23,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C
2,2025-09-24,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C
3,2025-09-25,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C
4,2025-09-26,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C


## Step 6: Lag and rolling features

For each SKU-day, look backward: what did this SKU do 1/2/3/7/14/21/28/364
days ago, its recent rolling average/volatility over 7/28/56 days, and how
many days since it was last picked at all. `shift(1)` is applied before
every rolling calculation so "today" never leaks into its own feature.

In [8]:
def add_lag_and_rolling_features(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.sort_values(["FROM_ITEM", "day"]).reset_index(drop=True)
    g = panel.groupby("FROM_ITEM")["pick_lines"]

    for lag in LAG_DAYS:
        panel[f"lag_{lag}"] = g.shift(lag)

    for window in ROLLING_WINDOWS:
        shifted = g.shift(1)
        panel[f"roll_mean_{window}"] = (
            shifted.groupby(panel["FROM_ITEM"]).rolling(window).mean().reset_index(level=0, drop=True)
        )
        panel[f"roll_std_{window}"] = (
            shifted.groupby(panel["FROM_ITEM"]).rolling(window).std().reset_index(level=0, drop=True)
        )

    panel["demand_cv_56"] = panel["roll_std_56"] / panel["roll_mean_56"].replace(0, np.nan)

    is_active = (panel["pick_lines"] > 0).astype(int)
    active_day_idx = panel["day"].map(pd.Timestamp.toordinal).to_numpy()
    panel["_active_day_num"] = np.where(is_active == 1, active_day_idx, np.nan)
    panel["_last_active_day_num"] = (
        panel.groupby("FROM_ITEM")["_active_day_num"].apply(lambda s: s.shift(1).ffill())
        .reset_index(level=0, drop=True)
    )
    panel["days_since_last_pick"] = active_day_idx - panel["_last_active_day_num"]
    panel = panel.drop(columns=["_active_day_num", "_last_active_day_num"])

    return panel

panel = add_lag_and_rolling_features(panel)
panel.head()

,day,FROM_ITEM,pick_lines,case_pick_lines,pallet_pull_lines,other_lines,FROM_DESCR,CASES_PER_PALLET,product_family,velocity_tier,...,lag_28,lag_364,roll_mean_7,roll_std_7,roll_mean_28,roll_std_28,roll_mean_56,roll_std_56,demand_cv_56,days_since_last_pick
0,2025-09-22,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-23,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-09-24,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-09-25,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-09-26,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 7: Calendar features (including day-of-week / weekend)

This is the whole reason the pipeline moved from weekly to daily grain: a
weekly total makes every day inside that week indistinguishable. Now the
model can see `day_of_week`, `day_name`, and `is_weekend` directly - and
sure enough, Saturdays/Sundays average noticeably fewer pick lines than
weekdays in this data.

In [9]:
def add_calendar_features(panel: pd.DataFrame) -> pd.DataFrame:
    day = panel["day"]
    panel["day_of_week"] = day.dt.dayofweek          # Monday=0 ... Sunday=6
    panel["day_name"] = day.dt.day_name()
    panel["is_weekend"] = day.dt.dayofweek.isin([5, 6]).astype(int)

    panel["week_of_year"] = day.dt.isocalendar().week.astype(int)
    panel["month"] = day.dt.month
    panel["quarter"] = day.dt.quarter
    panel["is_holiday_season"] = panel["month"].isin([11, 12]).astype(int)

    us_holidays = {
        "thanksgiving_day": [pd.Timestamp("2025-11-27"), pd.Timestamp("2026-11-26")],
        "christmas_day": [pd.Timestamp("2025-12-25"), pd.Timestamp("2026-12-25")],
    }
    for name, dates in us_holidays.items():
        panel[f"is_{name}"] = day.isin(dates).astype(int)

    return panel

panel = add_calendar_features(panel)
panel.head()

,day,FROM_ITEM,pick_lines,case_pick_lines,pallet_pull_lines,other_lines,FROM_DESCR,CASES_PER_PALLET,product_family,velocity_tier,...,days_since_last_pick,day_of_week,day_name,is_weekend,week_of_year,month,quarter,is_holiday_season,is_thanksgiving_day,is_christmas_day
0,2025-09-22,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,0,Monday,0,39,9,3,0,0,0
1,2025-09-23,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,1,Tuesday,0,39,9,3,0,0,0
2,2025-09-24,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,2,Wednesday,0,39,9,3,0,0,0
3,2025-09-25,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,3,Thursday,0,39,9,3,0,0,0
4,2025-09-26,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,NaN,4,Friday,0,39,9,3,0,0,0


## Step 8: Fiscal day/quarter numbering and the train/test split

Numbers every day sequentially from the start of your data, groups them
into 91-day fiscal quarters (13 weeks), and marks the most recent 91 days
as `test` and everything before as `train`. This stays correct
automatically as more data is added later.

In [10]:
def add_fiscal_split_labels(panel: pd.DataFrame) -> pd.DataFrame:
    day_order = sorted(panel["day"].unique())
    n_days = len(day_order)
    fiscal_day_num = {d: i + 1 for i, d in enumerate(day_order)}
    panel["fiscal_day"] = panel["day"].map(fiscal_day_num)
    panel["fiscal_week"] = ((panel["fiscal_day"] - 1) // 7) + 1
    panel["fiscal_year"] = ((panel["fiscal_day"] - 1) // 364) + 1
    panel["fiscal_quarter"] = (((panel["fiscal_day"] - 1) % 364) // 91) + 1
    panel["split"] = np.where(panel["fiscal_day"] > n_days - TEST_DAYS, "test", "train")
    return panel

panel = add_fiscal_split_labels(panel)
panel.head()

,day,FROM_ITEM,pick_lines,case_pick_lines,pallet_pull_lines,other_lines,FROM_DESCR,CASES_PER_PALLET,product_family,velocity_tier,...,month,quarter,is_holiday_season,is_thanksgiving_day,is_christmas_day,fiscal_day,fiscal_week,fiscal_year,fiscal_quarter,split
0,2025-09-22,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,9,3,0,0,0,1,1,1,1,train
1,2025-09-23,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,9,3,0,0,0,2,1,1,1,train
2,2025-09-24,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,9,3,0,0,0,3,1,1,1,train
3,2025-09-25,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,9,3,0,0,0,4,1,1,1,train
4,2025-09-26,20000035,0.0,0.0,0.0,0.0,AHBS FRIDGE FRSH 8/1CT,225,AHBS,C,...,9,3,0,0,0,5,1,1,1,train


## Step 9: Rename to shared column names and tag the source

Doesn't change any values - renames columns to the names the rest of the
team's code already uses (`item_id`, `date`, `n_lines`, etc), so this file
can be read directly without a manual column-mapping step, and adds a
`source` column so every row carries an honest label that this is still a
pick-based proxy for demand, not a real order-line extract.

In [11]:
def finalize_output_schema(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.rename(columns={
        "FROM_ITEM": "item_id",
        "FROM_DESCR": "item_descr",
        "CASES_PER_PALLET": "cases_per_pallet",
        "pick_lines": "n_lines",
        "day": "date",
    })

    panel["date"] = panel["date"].dt.date.astype(str)
    panel["source"] = SOURCE_LABEL

    lead_cols = ["item_id", "date", "day_of_week", "day_name", "is_weekend", "source",
                 "n_lines", "case_pick_lines", "pallet_pull_lines", "other_lines"]
    other_cols = [c for c in panel.columns if c not in lead_cols]
    return panel[lead_cols + other_cols]

panel = finalize_output_schema(panel)
panel.head()

,item_id,date,day_of_week,day_name,is_weekend,source,n_lines,case_pick_lines,pallet_pull_lines,other_lines,...,month,quarter,is_holiday_season,is_thanksgiving_day,is_christmas_day,fiscal_day,fiscal_week,fiscal_year,fiscal_quarter,split
0,20000035,2025-09-22,0,Monday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,1,1,1,1,train
1,20000035,2025-09-23,1,Tuesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,2,1,1,1,train
2,20000035,2025-09-24,2,Wednesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,3,1,1,1,train
3,20000035,2025-09-25,3,Thursday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,4,1,1,1,train
4,20000035,2025-09-26,4,Friday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,5,1,1,1,train


## Step 10: Save the finished panel

This is the file your teammate loads to train the model - filter on
`split` to get the train and test sets.

In [12]:
panel.to_csv(OUTPUT_FILE, index=False)

n_days = panel["date"].nunique()
n_test_days = panel.loc[panel["split"] == "test", "date"].nunique()
print(f"Saved {len(panel):,} rows, {panel.shape[1]} columns to {OUTPUT_FILE}")
print(f"Days covered: {n_days} ({n_days - n_test_days} train / {n_test_days} test) "
      f"| SKUs: {panel['item_id'].nunique()}")
print(f"\nSource: {SOURCE_LABEL}\n{CAVEAT}")

Saved 638,092 rows, 41 columns to sku_day_panel.csv
Days covered: 364 (273 train / 91 test) | SKUs: 1753

Source: pick_proxy_filtered
Every count in this file comes from WAREHOUSE PICK TRANSACTIONS, filtered to rows with a non-blank CUST_NAME (customer-order picks). This is a PROXY for real demand, not a direct order-line extract: the pick timestamp (CREATE_DT) marks when the warehouse released the work, not when the customer actually placed the order. Replace this source with a true order-line extract (an order-received timestamp) once one exists.


## Quick sanity check

Confirm the weekday/weekend pattern actually shows up (this is the whole
point of switching to daily grain), and look at one SKU's row across a few
days.

In [13]:
print(panel.groupby("day_name")["n_lines"].mean().sort_values(ascending=False))
print()
print(panel.groupby("is_weekend")["n_lines"].mean())

sample_sku = panel["item_id"].iloc[0]
panel[panel["item_id"] == sample_sku].head(10)

day_name
Tuesday      4.045219
Wednesday    3.952115
Thursday     3.869915
Friday       3.523783
Monday       3.340405
Saturday     2.515797
Sunday       2.119882
Name: n_lines, dtype: float64

is_weekend
0    3.746288
1    2.317840
Name: n_lines, dtype: float64


,item_id,date,day_of_week,day_name,is_weekend,source,n_lines,case_pick_lines,pallet_pull_lines,other_lines,...,month,quarter,is_holiday_season,is_thanksgiving_day,is_christmas_day,fiscal_day,fiscal_week,fiscal_year,fiscal_quarter,split
0,20000035,2025-09-22,0,Monday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,1,1,1,1,train
1,20000035,2025-09-23,1,Tuesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,2,1,1,1,train
2,20000035,2025-09-24,2,Wednesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,3,1,1,1,train
3,20000035,2025-09-25,3,Thursday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,4,1,1,1,train
4,20000035,2025-09-26,4,Friday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,5,1,1,1,train
5,20000035,2025-09-27,5,Saturday,1,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,6,1,1,1,train
6,20000035,2025-09-28,6,Sunday,1,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,7,1,1,1,train
7,20000035,2025-09-29,0,Monday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,8,2,1,1,train
8,20000035,2025-09-30,1,Tuesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,9,3,0,0,0,9,2,1,1,train
9,20000035,2025-10-01,2,Wednesday,0,pick_proxy_filtered,0.0,0.0,0.0,0.0,...,10,4,0,0,0,10,2,1,1,train
